# Emotion-Aware Public Speaking Coach: Feedback Generation Pipeline

## Introduction

This notebook implements the generation stage of the Emotion-Aware Public Speaking Coach. The objective is to transform raw presentation recordings into personalized coaching feedback by combining emotion recognition models with a large language model (LLM).

The pipeline consists of two sequential stages:

1. **Emotion Recognition**

   * Speech transcription using Whisper.
   * Text Emotion Recognition (TER) using a BERT-based classifier.
   * Speech Emotion Recognition (SER) using a Wav2Vec2-based classifier.
   * Construction of a multimodal emotion timeline.

2. **Feedback Generation**

   * Conversion of the emotion timeline into a textual representation.
   * Conditioning Gemma4 on the generated timeline.
   * Generation of coaching feedback under four prompt configurations:

     * Academic Advisor + Simple Prompting
     * Academic Advisor + Detailed Prompting
     * Pitch Mentor + Simple Prompting
     * Pitch Mentor + Detailed Prompting

For each presentation:

```text
Presentation Audio
    ↓
Whisper Transcription
    ↓
Segmented Transcript
    ↓
TER + SER
    ↓
Emotion Recognition
    ↓
Feedback Coach
```

The outputs generated in this notebook are subsequently evaluated in **2_llm_judges.ipynb** using an LLM-as-a-Judge protocol.

---

## Section 1 — Emotion Recognition

### Purpose

The goal of this stage is to transform presentation recordings into structured multimodal emotion timelines that can be interpreted by the coaching agent.

Each presentation is processed using three independent components:

* Automatic Speech Recognition (ASR)
* Text Emotion Recognition (TER)
* Speech Emotion Recognition (SER)

Emotion predictions are computed for each transcription segment and later merged into a timeline representation.

---

### Models Used

| Component | Model                                          |
| --------- | ---------------------------------------------- |
| ASR       | Whisper Base                                   |
| TER       | boltuix/bert-emotion                           |
| SER       | r-f/wav2vec-english-speech-emotion-recognition |

The TER model produces emotion probabilities from transcript segments, while the SER model predicts emotions directly from the corresponding audio segments.

For both modalities, the top-3 emotion predictions are retained.

---

### Output Structure

Each timeline entry contains:

```json
{
    "start": 12.5,
    "end": 18.2,
    "text": "...",
    "top3": [...]
}
```


In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm 
tqdm.pandas()

from emotion_recognition import EmotionRecognition

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def process_row(row, emo: EmotionRecognition):

    file_path = f"../data/dataset/{row['title']}.mp3"

    if not os.path.exists(file_path):
        return {
            "transcription": "",
            "text_emotion": "",
            "speech_emotion": ""
        }

    result = emo._process_audio(file_path)
    predictions = emo._emotion_recognition(result)

    return {
        "transcription": predictions["full_text"],
        "text_emotion": json.dumps(predictions["text_emotion"]),
        "speech_emotion": json.dumps(predictions["audio_emotion"])
    }

emo = EmotionRecognition()


In [12]:
df = pd.read_excel("../data/presentations.xlsx")

df[["transcription", "text_emotion", "speech_emotion"]] = (
    df.progress_apply(lambda row: pd.Series(process_row(row, emo)), axis=1)
)

df.to_excel("../data/output_predictions.xlsx", index=False)

df.to_json("../data/output_predictions.jsonl", orient="records", lines=True)

INFO:emotion_recognition:Audio processed: 27 segments extracted.
Current allocated memory: 3.54 GB
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO:emotion_recognition:Emotion recognition completed.
Current allocated memory: 3.73 GB
  8%|▊         | 2/24 [00:18<03:27,  9.45s/it][src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
INFO:emotion_recognition:Audio processed: 28 segments extracted.
Current allocated memory: 3.54 GB
INFO:emotion_recognition:Emotion recognition completed.
Current allocated memory: 3.62 GB
 12%|█▎        | 3/24 [00:30<03:36, 10.31s/it]INFO:emotion_recognition:Audio processed: 40 segments extracted.
Current allocated memory: 3.54 GB
INFO:emotion_recognition:Emotion recognition completed.
Current allocated memory: 3.62 GB
 17%|█▋        | 4/24 [00:45<04:02, 12.11s/it][src/libmpg123/id3.c:process_comment():587] error: No comment text

## Section 2 — Coaching Feedback Generation

### Purpose

The objective of this stage is to transform the emotion timeline into actionable presentation feedback, generated by **Gemma4:E4B**.

A visual guide to Gemma4: https://newsletter.maartengrootendorst.com/p/a-visual-guide-to-gemma-4

Rather than providing the full audio or transcript directly to the LLM, the model receives only the emotion timeline representation. This design allows us to evaluate how effectively emotion signals alone can guide presentation coaching.

---

### Coach Personas

Two coaching personas are evaluated.

#### Academic Advisor

Focuses on:

* argumentative clarity
* scientific rigor
* communication strategy
* presentation structure

#### Pitch Mentor

Focuses on:

* audience engagement
* narrative development
* persuasion
* storytelling effectiveness

Both personas use the same underlying language model and differ only through prompt conditioning.

---

### Prompting Conditions

We compare two prompting strategies.

#### Simple Prompting

Includes:

* role definition
* task specification

No explicit reasoning instructions or output formatting constraints are provided.

#### Detailed Prompting

Adds:

* behavioral guidelines
* reasoning instructions
* output structure constraints
* quality requirements

This allows us to evaluate the effect of prompt engineering on generated coaching feedback.

---

### Experimental Conditions

Each presentation produces four responses:

| Condition         | Persona          | Prompt Type |
| ----------------- | ---------------- | ----------- |
| Academic-Simple   | Academic Advisor | Simple      |
| Academic-Detailed | Academic Advisor | Detailed    |
| Pitch-Simple      | Pitch Mentor     | Simple      |
| Pitch-Detailed    | Pitch Mentor     | Detailed    |

For 24 presentations this yields:

```text
24 presentations × 4 conditions = 96 generated responses
```

---

### Outputs

This section exports:

```text
output_predictions.jsonl
```

Containing:

* transcript
* text emotions
* speech emotions

and

```text
llm_evaluations_gemma4.xlsx
```

Containing:

* emotion timeline
* Academic-Simple feedback
* Academic-Detailed feedback
* Pitch-Simple feedback
* Pitch-Detailed feedback

These outputs serve as inputs for the evaluation phase in **2_llm_judges.ipynb**.



In [ ]:
from feedback_coach import generate_llm_evaluations

generate_llm_evaluations(
    input_jsonl="../data/output_predictions.jsonl",
    output_excel="../data/llm_evaluations_gemma4.xlsx",
    api_host="http://localhost:11434", # Ollama port
    model='ollama_chat/gemma4:e4b'
)


  0%|          | 0/24 [00:00<?, ?it/s]

[0.0s - 12.0s] Audio: angry (0.15) | Text: neutral (0.58) | Transcript: "Our fourth participant is Jordan Minor from our Graduate School of Biomedical Science and Engineering."
[21.0s - 25.0s] Audio: angry (0.16) | Text: neutral (0.66) | Transcript: "Imagine you've just been diagnosed with early stage press cancer."
[25.0s - 28.0s] Audio: angry (0.16) | Text: confusion (0.60) | Transcript: "What are the immediate emotions running through your mind?"
[28.0s - 30.0s] Audio: angry (0.16) | Text: fear (0.92) | Transcript: "Scared, stressed, anxious?"
[30.0s - 36.0s] Audio: surprise (0.15) | Text: confusion (0.70) | Transcript: "Now what if I tell you with an 88% chance of living in other 10 years if you do absolutely nothing?"
[36.0s - 38.0s] Audio: happy (0.16) | Text: neutral (0.55) | Transcript: "Would you seek treatment?"
[38.0s - 40.0s] Audio: happy (0.15) | Text: neutral (0.70) | Transcript: "Or would you take the risk?"
[40.0s - 45.0s] Audio: angry (0.15) | Text: neutral (0.53) | Tr

,n_record,youtube_url,event,title,category,transcription,text_emotion,speech_emotion,emotion_timeline,academic_simple,academic_detailed,pitch_simple,pitch_detailed
0,0,https://www.youtube.com/watch?v=KuwkL6qaoYo,3MT,3MT_2024_0,academic,Our fourth participant is Jordan Minor from ou...,"[{""start"": 0.0, ""end"": 12.0, ""text"": ""Our four...","[{""start"": 0.0, ""end"": 12.0, ""top3"": [{""score""...",[0.0s - 12.0s] Audio: angry (0.15) | Text: neu...,The presentation demonstrates a high degree of...,**Strengths:**\nThe presentation excels at est...,The core scientific narrative and the stakes y...,"**Strengths:**\nThe presentation has a strong,..."
1,1,https://www.youtube.com/watch?v=mM-tdBI1l3c,3MT,3MT_2024_1,academic,"In our final presenter today is Liza White, wh...","[{""start"": 0.0, ""end"": 10.16, ""text"": ""In our ...","[{""start"": 0.0, ""end"": 10.16, ""top3"": [{""score...",[0.0s - 10.2s] Audio: angry (0.16) | Text: hap...,The presentation demonstrates a strong foundat...,Strengths:\n- **Strong Narrative Arc and Stake...,The structure of your presentation is highly e...,**Strengths:**\nThe narrative structure and em...
2,2,https://www.youtube.com/watch?v=FV1THB2HeHM,3MT,3MT_2024_2,academic,So I'm very pleased to introduce our first con...,"[{""start"": 0.0, ""end"": 13.64, ""text"": ""So I'm ...","[{""start"": 0.0, ""end"": 13.64, ""top3"": [{""score...",[0.0s - 13.6s] Audio: fear (0.15) | Text: happ...,The presentation possesses a deeply compelling...,Strengths:\n- **Compelling Narrative Arc:** Th...,The emotional delivery of this presentation co...,**Strengths:**\n* **Strong Narrative Arc:** ...
3,3,https://www.youtube.com/watch?v=HZkB2dS-KcU,3MT,3MT_2024_3,academic,The biology is not just about the things you c...,"[{""start"": 0.0, ""end"": 14.0, ""text"": ""The biol...","[{""start"": 0.0, ""end"": 14.0, ""top3"": [{""score""...",[0.0s - 14.0s] Audio: angry (0.16) | Text: neu...,"The presentation demonstrates a strong, compel...",Strengths:\n- **Conceptual Depth and Rigor:** ...,To maximize the persuasive impact of this pres...,**Strengths:**\nThe narrative structure is hig...
4,4,https://www.youtube.com/watch?v=PFFZpiZ3ELo,3MT,3MT_2024_4,academic,"Our fifth presenter is Sean Cibli, also from t...","[{""start"": 0.0, ""end"": 11.120000000000001, ""te...","[{""start"": 0.0, ""end"": 11.120000000000001, ""to...",[0.0s - 11.1s] Audio: happy (0.15) | Text: neu...,The presentation demonstrates a highly engagin...,Strengths:\n- **Strong Narrative Arc:** The pr...,The structure of your presentation is highly e...,**Strengths:**\nThe presentation structure is ...
5,5,https://www.youtube.com/watch?v=Dk5AgDej4fk,3MT,3MT_2024_5,academic,Our third contestant from the main college of ...,"[{""start"": 0.0, ""end"": 10.88, ""text"": ""Our thi...","[{""start"": 0.0, ""end"": 10.88, ""top3"": [{""score...",[0.0s - 10.9s] Audio: happy (0.15) | Text: hap...,The presentation demonstrates a strong foundat...,Strengths:\n- **Rigor of Argument:** The prese...,The core content of your research is highly im...,Strengths:\n- **Effective Problem Framing:** T...
6,6,https://www.youtube.com/watch?v=SPE-K1XAmD0,3MT,3MT_2025_0,academic,This is Tiasha Shevo-Rich. She's a co-detail P...,"[{""start"": 0.0, ""end"": 6.32, ""text"": ""This is ...","[{""start"": 0.0, ""end"": 6.32, ""top3"": [{""score""...",[0.0s - 6.3s] Audio: happy (0.16) | Text: neut...,The presentation demonstrates a highly effecti...,Strengths:\n- **Masterful Narrative Arc:** The...,The presentation has a powerful narrative stru...,**Strengths:**\nThe presentation excels at bui...
7,7,https://www.youtube.com/watch?v=gEr_tdxjnZE,3MT,3MT_2025_1,academic,Our next speaker is Limna Suja Shaji. Limna is...,"[{""start"": 0.0, ""end"": 9.24, ""text"": ""Our next...","[{""start"": 0.0, ""end"": 9.24, ""top3"": [{""score""...",[0.0s - 9.2s] Audio: fear (0.15) | Text: love ...,The presentation demonstrates a highly compell...,Strengths:\n- **Scientific Depth and Rigor:** ...,The presen